# Short term data preparation for event study

# Tweets and financial markets

## Group Project

Cathy (Cui) Yu, Hefei Mao, Yichi Wang, Di Yang, Marc Hayes

### Imports

In [1]:
# Standard library
from datetime import datetime, date, time as dtime, timedelta
import glob
import os
import re
from pathlib import Path
from time import time
from typing import Dict, List, Tuple, Optional

# Third-party
import numpy as np
import pandas as pd
from pandas.errors import ParserError
import pytz

import warnings
# optional: quiet PerformanceWarning globally (use only if you still see stray warnings)
# warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

### Loading the twitter data

In [2]:
df_tweets = pd.read_csv("tweets_event_windows_final.csv")
df_tweets.head()

,new_id,id,text,date,event_minute,ticker,company_name,industry,event_time_et,in_trading_hours,adjusted_event_time_et,adjusted_event_time_utc,event_1h_pre_et,event_1h_end_et,event_5d_pre_et,event_5d_end_et,event_1h_pre,event_1h_end,event_5d_pre,event_5d_end
0,52,911287725847908352,Thank you to Doug Parker and American Airlines...,2017-09-22 17:54:59,2017-09-22 17:54:00+00:00,AAL,American Airlines Group,Airlines & Travel,2017-09-22 13:54:00-04:00,0,2017-09-22 13:54:00-04:00,2017-09-22 17:54:00+00:00,2017-09-22 12:54:00-04:00,2017-09-22 14:54:00-04:00,2017-09-15 09:30:00-04:00,2017-09-29 09:30:00-04:00,2017-09-22 16:54:00+00:00,2017-09-22 18:54:00+00:00,2017-09-15 13:30:00+00:00,2017-09-29 13:30:00+00:00
1,149,1184147319480041473,"Join me in Dallas, Texas this Thursday (Octobe...",2019-10-15 16:41:36,2019-10-15 16:41:00+00:00,AAL,American Airlines Group,Airlines & Travel,2019-10-15 12:41:00-04:00,0,2019-10-15 12:41:00-04:00,2019-10-15 16:41:00+00:00,2019-10-15 11:41:00-04:00,2019-10-15 13:41:00-04:00,2019-10-08 09:30:00-04:00,2019-10-22 09:30:00-04:00,2019-10-15 15:41:00+00:00,2019-10-15 17:41:00+00:00,2019-10-08 13:30:00+00:00,2019-10-22 13:30:00+00:00
2,148,1184631273454817280,"THANK YOU you Dallas, Texas. See you tomorrow ...",2019-10-17 00:44:39,2019-10-17 00:44:00+00:00,AAL,American Airlines Group,Airlines & Travel,2019-10-16 20:44:00-04:00,1,2019-10-17 09:30:00-04:00,2019-10-17 13:30:00+00:00,2019-10-17 08:30:00-04:00,2019-10-17 10:30:00-04:00,2019-10-10 09:30:00-04:00,2019-10-24 09:30:00-04:00,2019-10-17 12:30:00+00:00,2019-10-17 14:30:00+00:00,2019-10-10 13:30:00+00:00,2019-10-24 13:30:00+00:00
3,147,1184987864125321216,Just arrived at the American Airlines Center i...,2019-10-18 00:21:37,2019-10-18 00:21:00+00:00,AAL,American Airlines Group,Airlines & Travel,2019-10-17 20:21:00-04:00,1,2019-10-18 09:30:00-04:00,2019-10-18 13:30:00+00:00,2019-10-18 08:30:00-04:00,2019-10-18 10:30:00-04:00,2019-10-11 09:30:00-04:00,2019-10-25 09:30:00-04:00,2019-10-18 12:30:00+00:00,2019-10-18 14:30:00+00:00,2019-10-11 13:30:00+00:00,2019-10-25 13:30:00+00:00
4,24,700795170023825408,"I use both iPhone &amp, Samsung. If Apple does...",2016-02-19 21:32:43,2016-02-19 21:32:00+00:00,AAPL,Apple Inc.,Technology & Internet,2016-02-19 16:32:00-05:00,1,2016-02-22 09:30:00-05:00,2016-02-22 14:30:00+00:00,2016-02-22 08:30:00-05:00,2016-02-22 10:30:00-05:00,2016-02-12 09:30:00-05:00,2016-02-29 09:30:00-05:00,2016-02-22 13:30:00+00:00,2016-02-22 15:30:00+00:00,2016-02-12 14:30:00+00:00,2016-02-29 14:30:00+00:00


### Preprocessing twitter data

In [3]:
# For the short term analysis we focus on the minutely data, we decided to use Eastern Time

# Convert the relevant time columns from object type to pandas timestamp from UTC to ET
cols = ["event_time_et", "event_1h_pre_et", "event_1h_end_et"]

for col in cols:
    df_tweets[col] = (
        pd.to_datetime(df_tweets[col], utc=True, errors="coerce")
          .dt.tz_convert("America/New_York")
    )

# Convert "date" column to timestamp, but keep it in UTC -> "date_utc"
df_tweets["date_utc"] = pd.to_datetime(df_tweets["date"], utc=True, errors="coerce")

### Filtering the tweets to retain only relevant entries

In [4]:
df_tweets.columns

Index(['new_id', 'id', 'text', 'date', 'event_minute', 'ticker',
       'company_name', 'industry', 'event_time_et', 'in_trading_hours',
       'adjusted_event_time_et', 'adjusted_event_time_utc', 'event_1h_pre_et',
       'event_1h_end_et', 'event_5d_pre_et', 'event_5d_end_et', 'event_1h_pre',
       'event_1h_end', 'event_5d_pre', 'event_5d_end', 'date_utc'],
      dtype='object')

In [5]:
# Keep only the relevant columns for the long term analysis
df_tweets_filtered = df_tweets[[
    "id", 
    "text", 
    "date_utc",
    "event_time_et", 
    # "event_1h_pre_et", 
    # "event_1h_end_et", 
    "ticker", 
    "company_name", 
    "industry"
]]
df_tweets_filtered.head()

,id,text,date_utc,event_time_et,ticker,company_name,industry
0,911287725847908352,Thank you to Doug Parker and American Airlines...,2017-09-22 17:54:59+00:00,2017-09-22 13:54:00-04:00,AAL,American Airlines Group,Airlines & Travel
1,1184147319480041473,"Join me in Dallas, Texas this Thursday (Octobe...",2019-10-15 16:41:36+00:00,2019-10-15 12:41:00-04:00,AAL,American Airlines Group,Airlines & Travel
2,1184631273454817280,"THANK YOU you Dallas, Texas. See you tomorrow ...",2019-10-17 00:44:39+00:00,2019-10-16 20:44:00-04:00,AAL,American Airlines Group,Airlines & Travel
3,1184987864125321216,Just arrived at the American Airlines Center i...,2019-10-18 00:21:37+00:00,2019-10-17 20:21:00-04:00,AAL,American Airlines Group,Airlines & Travel
4,700795170023825408,"I use both iPhone &amp, Samsung. If Apple does...",2016-02-19 21:32:43+00:00,2016-02-19 16:32:00-05:00,AAPL,Apple Inc.,Technology & Internet


### Processing the stock data

### Minutely data for main trading hours (ET 9:30 - 16:00, but in UTC)

In [6]:
# NOTE: Data preprocessing, only needs to run once and can then be commented out.
# For submission this cell is interrupted to save runtime.

# Build minutely files with returns, log returns, expected return, and abnormal return

# ---------- USER CONFIG ----------
INPUT_DIR = "DATA EOD"
OUTPUT_DIR = "DATA MINUTELY"
os.makedirs(OUTPUT_DIR, exist_ok=True)

START_DATE = date(2015, 12, 1)
END_DATE   = date(2020, 1, 31)

ET = pytz.timezone("US/Eastern")
SESSION_OPEN_ET  = (9, 30)
SESSION_CLOSE_ET = (16, 0)

HIGH_LOW_FILL = "nan"   # options: "nan", "zero", "ffill"
DROP_OUTSIDE_SESSION = True

# Expected-return window (minutes). Moving average of the previous N minutes (default 60 = 1 hour).
EXPECTED_WINDOW_MINS = 60
# ---------------------------------

def et_session_range_for_day(day: date):
    dt_open_et = datetime(day.year, day.month, day.day, SESSION_OPEN_ET[0], SESSION_OPEN_ET[1])
    dt_close_et = datetime(day.year, day.month, day.day, SESSION_CLOSE_ET[0], SESSION_CLOSE_ET[1])
    dt_open_et = ET.localize(dt_open_et)
    dt_close_et = ET.localize(dt_close_et)
    return dt_open_et.astimezone(pytz.UTC), dt_close_et.astimezone(pytz.UTC)

def build_all_session_minutes(start_date: date, end_date: date):
    all_minutes = []
    cur = start_date
    while cur <= end_date:
        if cur.weekday() < 5:  # Mon-Fri (no holiday handling)
            open_utc, close_utc = et_session_range_for_day(cur)
            minutes = pd.date_range(start=open_utc, end=close_utc, freq="min", tz=pytz.UTC)
            all_minutes.append(minutes)
        cur += timedelta(days=1)
    if not all_minutes:
        return pd.DatetimeIndex([], tz=pytz.UTC)
    full_index = all_minutes[0]
    for idx in all_minutes[1:]:
        full_index = full_index.union(idx)
    return full_index

def process_file_minutely(input_path, output_path):
    print(f"Processing {input_path}")

    # read file robustly
    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        print(f"  Error reading file → skipped: {e}")
        return

    if "timestamp" not in df.columns:
        print("  No 'timestamp' column → skipped.")
        return

    # parse timestamps as UTC
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    n_bad = df["timestamp"].isna().sum()
    if n_bad:
        print(f"  Warning: {n_bad} rows have invalid timestamps and will be dropped.")
        df = df.dropna(subset=["timestamp"])

    # ---- DATE RANGE FILTER ----
    df["date_utc"] = df["timestamp"].dt.date
    df = df[(df["date_utc"] >= START_DATE) & (df["date_utc"] <= END_DATE)].copy()
    df.drop(columns=["date_utc"], inplace=True)

    if df.empty:
        print("  No data found inside specified date range → skipped.")
        return

    # ---- SESSION FILTER ----
    if DROP_OUTSIDE_SESSION:
        df["timestamp_et"] = df["timestamp"].dt.tz_convert(ET)
        def in_session(ts_et):
            tod = (ts_et.hour, ts_et.minute)
            return (tod >= SESSION_OPEN_ET) and (tod <= SESSION_CLOSE_ET)
        df = df[df["timestamp_et"].apply(in_session)].copy()
        df.drop(columns=["timestamp_et"], inplace=True)

        if df.empty:
            print("  No data inside trading session window → skipped.")
            return

    # sort and set index
    df = df.sort_values("timestamp").set_index("timestamp")

    # build/reuse global minutes index
    global __ALL_SESSION_MINUTES_CACHE
    if "__ALL_SESSION_MINUTES_CACHE" not in globals():
        __ALL_SESSION_MINUTES_CACHE = build_all_session_minutes(START_DATE, END_DATE)
    full_minutes = __ALL_SESSION_MINUTES_CACHE

    # restrict minutes slice to overlap between file data and global index for speed
    mask = (full_minutes >= df.index.min().floor("D")) & (full_minutes <= df.index.max().ceil("D"))
    minutes_slice = full_minutes[mask]
    if minutes_slice.empty:
        print("  No session minutes overlap with this file's timestamps in the requested date range → skipped.")
        return

    reindexed = df.reindex(minutes_slice)

    # Ensure expected columns exist
    for col in ["open", "high", "low", "close", "volume"]:
        if col not in reindexed.columns:
            reindexed[col] = np.nan

    # Forward-fill close so last-known price persists
    reindexed["close"] = reindexed["close"].ffill()

    # For missing open, set equal to last-known close (ffilled)
    open_missing = reindexed["open"].isna()
    reindexed.loc[open_missing, "open"] = reindexed.loc[open_missing, "close"]

    # Missing volumes -> 0
    reindexed["volume"] = reindexed["volume"].fillna(0)

    # High/Low behavior
    if HIGH_LOW_FILL == "zero":
        reindexed["high"] = reindexed["high"].fillna(0)
        reindexed["low"] = reindexed["low"].fillna(0)
    elif HIGH_LOW_FILL == "ffill":
        reindexed["high"] = reindexed["high"].fillna(reindexed["close"])
        reindexed["low"] = reindexed["low"].fillna(reindexed["close"])
    # else keep NaNs

    # Typical price
    reindexed["typical_price"] = (reindexed["high"] + reindexed["low"] + reindexed["close"]) / 3.0

    # ------------------------
    # Returns / Expected / AR
    # ------------------------
    # Group by ET-local trading-day to avoid crossing sessions (DST-aware)
    et_dates = reindexed.index.tz_convert(ET).date
    # Use groupby with the et_dates series as grouper
    grouped = reindexed.groupby(et_dates)

    # actual simple return (named 'return') and log return (named 'log_return')
    # compute per-group to avoid crossing session boundaries
    def compute_returns_for_group(g):
        # g is a DataFrame slice for one session (index = UTC minute timestamps)
        close = g["close"]
        simple_ret = close.pct_change()                    # (close_t / close_{t-1}) - 1
        log_ret = np.log(close / close.shift(1))          # log returns (NaN safe if divide by zero or NaN)
        return pd.DataFrame({"return": simple_ret, "log_return": log_ret})

    returns_parts = grouped.apply(lambda g: compute_returns_for_group(g)).reset_index(level=0, drop=True)
    # Align indices
    returns_parts = returns_parts.reindex(reindexed.index)

    # expected_return: moving average of previous EXPECTED_WINDOW_MINS simple returns (shifted so current excluded)
    # Compute per-group (session)
    def compute_expected(series_simple_ret):
        # shift(1) to exclude current minute, rolling mean over previous N minutes
        return series_simple_ret.shift(1).rolling(window=EXPECTED_WINDOW_MINS, min_periods=1).mean()

    expected_parts = grouped["close"].apply(lambda s: compute_expected(s.pct_change())).reset_index(level=0, drop=True)
    expected_parts = expected_parts.reindex(reindexed.index)

    # abnormal return (AR) = actual simple return - expected_return
    abnormal = returns_parts["return"] - expected_parts

    # Attach to reindexed
    reindexed["return"] = returns_parts["return"]
    reindexed["log_return"] = returns_parts["log_return"]
    reindexed["expected_return"] = expected_parts
    reindexed["abnormal_return"] = abnormal

    # Finalize and write CSV
    reindexed.index.name = "timestamp"
    out = reindexed.reset_index()
    out.to_csv(output_path, index=False)
    print(f"  Wrote {len(out)} minute rows to {output_path}")

def main_process_all_files():
    # Exclude tickers.csv explicitly
    csv_files = [
        f for f in glob.glob(os.path.join(INPUT_DIR, "*.csv"))
        if os.path.basename(f).lower() != "tickers.csv"
    ]

    if not csv_files:
        print("No CSV files found in INPUT_DIR.")
        return

    print("Building full session minute index...")
    global __ALL_SESSION_MINUTES_CACHE
    __ALL_SESSION_MINUTES_CACHE = build_all_session_minutes(START_DATE, END_DATE)
    print(f"  Built {len(__ALL_SESSION_MINUTES_CACHE)} total session minutes.")

    for path in csv_files:
        fname = os.path.basename(path)
        out_name = f"{os.path.splitext(fname)[0]}_minutely.csv"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        process_file_minutely(path, out_path)

# Run in notebook cell
main_process_all_files()

Building full session minute index...
  Built 425799 total session minutes.
Processing DATA EOD\AAL.csv
  Wrote 425799 minute rows to DATA MINUTELY\AAL_minutely.csv
Processing DATA EOD\AAPL.csv
  Wrote 425799 minute rows to DATA MINUTELY\AAPL_minutely.csv
Processing DATA EOD\AMD.csv
  Wrote 425799 minute rows to DATA MINUTELY\AMD_minutely.csv
Processing DATA EOD\AMZN.csv
  Wrote 425799 minute rows to DATA MINUTELY\AMZN_minutely.csv
Processing DATA EOD\APD.csv
  Wrote 425799 minute rows to DATA MINUTELY\APD_minutely.csv
Processing DATA EOD\APTV.csv
  Wrote 220524 minute rows to DATA MINUTELY\APTV_minutely.csv
Processing DATA EOD\AVGO.csv
  Wrote 425799 minute rows to DATA MINUTELY\AVGO_minutely.csv
Processing DATA EOD\BA.csv
  Wrote 425799 minute rows to DATA MINUTELY\BA_minutely.csv
Processing DATA EOD\BAC.csv


KeyboardInterrupt: 

In [7]:
df_amd_minutely = pd.read_csv("DATA MINUTELY/AMD_minutely.csv")
df_amd_minutely

,timestamp,Unnamed: 0,gmtoffset,datetime,open,high,low,close,volume,typical_price,return,log_return,expected_return,abnormal_return
0,2015-12-01 14:30:00+00:00,680.0,0.0,2015-12-01 14:30:00,2.360,2.3600,2.340,2.3599,114216.0,2.353300,NaN,NaN,NaN,NaN
1,2015-12-01 14:31:00+00:00,681.0,0.0,2015-12-01 14:31:00,2.350,2.3700,2.350,2.3600,28466.0,2.360000,0.000042,0.000042,NaN,NaN
2,2015-12-01 14:32:00+00:00,682.0,0.0,2015-12-01 14:32:00,2.365,2.3700,2.360,2.3650,7634.0,2.365000,0.002119,0.002116,0.000042,0.002076
3,2015-12-01 14:33:00+00:00,683.0,0.0,2015-12-01 14:33:00,2.370,2.3700,2.360,2.3650,6140.0,2.365000,0.000000,0.000000,0.001081,-0.001081
4,2015-12-01 14:34:00+00:00,684.0,0.0,2015-12-01 14:34:00,2.365,2.3699,2.360,2.3600,4775.0,2.363300,-0.002114,-0.002116,0.000720,-0.002835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425794,2020-01-31 20:56:00+00:00,42649.0,0.0,2020-01-31 20:56:00,47.040,47.0600,46.980,47.0150,330871.0,47.018333,-0.000638,-0.000638,-0.000193,-0.000445
425795,2020-01-31 20:57:00+00:00,42650.0,0.0,2020-01-31 20:57:00,47.015,47.0800,47.015,47.0650,294230.0,47.053333,0.001063,0.001063,-0.000205,0.001268
425796,2020-01-31 20:58:00+00:00,42651.0,0.0,2020-01-31 20:58:00,47.070,47.0800,47.010,47.0350,413844.0,47.041667,-0.000637,-0.000638,-0.000188,-0.000450
425797,2020-01-31 20:59:00+00:00,42652.0,0.0,2020-01-31 20:59:00,47.030,47.0400,46.960,46.9700,701973.0,46.990000,-0.001382,-0.001383,-0.000177,-0.001205


### Retrieving the estimate + event window

#### For all 50 companies / tickers

In [8]:
# Build 121-minute event panels for tweets

# ---------- CONFIG ----------
MINUTELY_DIR = "DATA MINUTELY"                                  # where <TICKER>_minutely.csv live
OUTPUT_DIR_COMPANY = "MINUTELY EVENT STUDY PANELS BY COMPANY"   # where per-event panels will be written
os.makedirs(OUTPUT_DIR_COMPANY, exist_ok=True)

# ET session (DST-aware)
ET = pytz.timezone("US/Eastern")
SESSION_OPEN_ET = (9, 30)
SESSION_CLOSE_ET = (16, 0)

# default window: 60 before, 60 after -> 121 minutes total
N_BEFORE = 60
N_AFTER = 60

# columns to extract from per-ticker minutely files (they will be prefixed by ticker_)
TICKER_COLUMNS = [
    "open", "high", "low", "close", "volume",
    "return", "log_return", "expected_return", "abnormal_return", "typical_price"
]
# --------------------------------

def et_session_range_for_day(day: date) -> Tuple[datetime, datetime]:
    dt_open_et = datetime(day.year, day.month, day.day, SESSION_OPEN_ET[0], SESSION_OPEN_ET[1])
    dt_close_et = datetime(day.year, day.month, day.day, SESSION_CLOSE_ET[0], SESSION_CLOSE_ET[1])
    dt_open_et = ET.localize(dt_open_et)
    dt_close_et = ET.localize(dt_close_et)
    return dt_open_et.astimezone(pytz.UTC), dt_close_et.astimezone(pytz.UTC)

# cache for session minutes for ranges
__SESSION_MINUTES_CACHE = {}

def build_session_minutes(start_date: date, end_date: date, force_rebuild: bool=False) -> pd.DatetimeIndex:
    """Return UTC DatetimeIndex for every trading-session minute between start_date and end_date inclusive.
       Uses a safe union loop (compatible with older pandas versions)."""
    key = (start_date.isoformat(), end_date.isoformat())
    if (not force_rebuild) and (key in __SESSION_MINUTES_CACHE):
        return __SESSION_MINUTES_CACHE[key]

    all_minutes = []
    cur = start_date
    while cur <= end_date:
        if cur.weekday() < 5:
            open_utc, close_utc = et_session_range_for_day(cur)
            minutes = pd.date_range(start=open_utc, end=close_utc, freq="min", tz=pytz.UTC)
            all_minutes.append(minutes)
        cur += timedelta(days=1)

    if not all_minutes:
        idx = pd.DatetimeIndex([], tz=pytz.UTC)
    else:
        # Safe union loop (works across pandas versions)
        idx = all_minutes[0]
        for mi in all_minutes[1:]:
            idx = idx.union(mi)
        # ensure sorted unique (union should already do this)
        idx = pd.DatetimeIndex(sorted(idx))
        # maintain tz
        if idx.tz is None:
            idx = idx.tz_localize(pytz.UTC)

    __SESSION_MINUTES_CACHE[key] = idx
    return idx

def _find_centered_window(session_index: pd.DatetimeIndex, event_ts_utc: pd.Timestamp,
                          n_before: int=N_BEFORE, n_after: int=N_AFTER) -> pd.DatetimeIndex:
    if len(session_index) == 0:
        return pd.DatetimeIndex([], tz=pytz.UTC)
    event_ts_utc = pd.to_datetime(event_ts_utc, utc=True)
    pos = session_index.searchsorted(event_ts_utc, side="left")
    before_start = max(0, pos - n_before)
    before_idx = session_index[before_start:pos]
    after_end = min(len(session_index), pos + n_after)
    after_idx = session_index[pos:after_end]
    return before_idx.append(after_idx)

def load_minutely_df(ticker: str, minutely_dir: str=MINUTELY_DIR) -> Optional[pd.DataFrame]:
    path = os.path.join(minutely_dir, f"{ticker}_minutely.csv")
    if not os.path.exists(path):
        found = glob.glob(os.path.join(minutely_dir, f"{ticker}*_minutely.csv"))
        if found:
            path = found[0]
        else:
            return None
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"  Error reading {path}: {e}")
        return None
    if "timestamp" not in df.columns:
        print(f"  File {path} has no 'timestamp' column; skipping.")
        return None
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp"]).set_index("timestamp").sort_index()
    return df

def _collect_ticker_list(minutely_dir: str=MINUTELY_DIR, tickers: Optional[List[str]]=None) -> List[str]:
    if tickers:
        return tickers
    files = glob.glob(os.path.join(minutely_dir, "*_minutely.csv"))
    tickers_found = []
    for p in files:
        bn = os.path.basename(p)
        if bn.lower().endswith("_minutely.csv"):
            ticker = bn[:-len("_minutely.csv")]
            tickers_found.append(ticker)
    tickers_found = sorted(set(tickers_found))
    return tickers_found

def build_minutely_event_panel(tweet_id: int,
                               df_tweets: pd.DataFrame,
                               event_ts_col: str = "event_ts_utc",
                               minutely_dir: str = MINUTELY_DIR,
                               tickers: Optional[List[str]] = None,
                               session_start_date: Optional[date] = None,
                               session_end_date: Optional[date] = None,
                               n_before: int = N_BEFORE,
                               n_after: int = N_AFTER,
                               save_csv: bool = True,
                               output_dir: str = OUTPUT_DIR_COMPANY) -> Tuple[pd.DataFrame, dict]:
    # 1) find tweet timestamp
    sel = df_tweets.loc[df_tweets["id"] == tweet_id, event_ts_col]
    if len(sel) == 0:
        raise KeyError(f"Tweet id {tweet_id} not found in df_tweets (expected column 'id').")
    event_ts = pd.to_datetime(sel.iloc[0], utc=True, errors="coerce")
    if pd.isna(event_ts):
        raise ValueError(f"Event timestamp for tweet {tweet_id} could not be parsed.")

    # 2) session index range
    if session_start_date is None or session_end_date is None:
        ev_date = event_ts.date()
        session_start_date = ev_date - timedelta(days=7)
        session_end_date = ev_date + timedelta(days=7)

    session_idx = build_session_minutes(session_start_date, session_end_date)
    if len(session_idx) == 0:
        raise RuntimeError("Built empty session index for provided date range.")

    # 3) compute desired (centered) timestamps of length n_before + n_after + 1
    pos = session_idx.searchsorted(event_ts, side="left")
    desired_positions = [pos - n_before + i for i in range(n_before + n_after + 1)]
    desired_timestamps = []
    for p in desired_positions:
        if 0 <= p < len(session_idx):
            desired_timestamps.append(session_idx[p])
        else:
            desired_timestamps.append(pd.NaT)

    panel_df = pd.DataFrame({"t": list(range(-n_before, n_after+1)), "timestamp": desired_timestamps})

    # 4) collect tickers
    ticker_list = _collect_ticker_list(minutely_dir, tickers)
    processed = []
    skipped = []

    for ticker in ticker_list:
        df_tick = load_minutely_df(ticker, minutely_dir=minutely_dir)
        if df_tick is None or df_tick.empty:
            skipped.append((ticker, "missing_or_empty"))
            # build empty columns in one go to avoid repeated inserts
            empty_cols = {f"{ticker}_{col}": [np.nan] * len(panel_df) for col in TICKER_COLUMNS}
            if empty_cols:
                panel_df = pd.concat([panel_df, pd.DataFrame(empty_cols, index=panel_df.index)], axis=1)
            continue

        vals_dict = {col: [] for col in TICKER_COLUMNS}
        for ts in desired_timestamps:
            if pd.isna(ts):
                for col in TICKER_COLUMNS:
                    vals_dict[col].append(np.nan)
                continue
            if ts in df_tick.index:
                row = df_tick.loc[ts]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                for col in TICKER_COLUMNS:
                    vals_dict[col].append(row[col] if col in row.index else np.nan)
            else:
                for col in TICKER_COLUMNS:
                    vals_dict[col].append(np.nan)

        # build a small DataFrame for this ticker and concat once
        ticker_df = pd.DataFrame(vals_dict, index=panel_df.index)
        # rename columns to include ticker prefix
        ticker_df = ticker_df.rename(columns={c: f"{ticker}_{c}" for c in ticker_df.columns})
        panel_df = pd.concat([panel_df, ticker_df], axis=1)

        processed.append(ticker)
    
    # force a copy to defragment the DataFrame memory layout and avoid future PerformanceWarnings
    panel_df = panel_df.copy()

    info = {
        "tweet_id": tweet_id,
        "event_ts": event_ts.isoformat(),
        "session_start_date": session_start_date.isoformat(),
        "session_end_date": session_end_date.isoformat(),
        "n_tickers_processed": len(processed),
        "n_tickers_skipped": len(skipped),
        "skipped": skipped,
        "n_rows": len(panel_df),
        "t_range": ( -n_before, n_after )
    }

    # 5) save CSV if requested
    if save_csv:
        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f"{tweet_id}_minutely_panel.csv")
        df_to_save = panel_df.copy()
        df_to_save["timestamp"] = df_to_save["timestamp"].apply(lambda x: x.isoformat() if pd.notna(x) else "")
        df_to_save.to_csv(out_path, index=False)
        info["saved_to"] = out_path
        print(f"Saved panel for tweet {tweet_id} to {out_path}")

    return panel_df, info

In [9]:
# Example quick test (uncomment & adapt):

panel, info = build_minutely_event_panel(
    tweet_id=df_tweets_filtered["id"].iloc[0], 
    df_tweets=df_tweets_filtered, 
    event_ts_col="date_utc",
    minutely_dir="DATA MINUTELY", session_start_date=date(2015,12,1),
    session_end_date=date(2020,1,31), n_before=60, n_after=60,
    save_csv=False
)

panel

,t,timestamp,AAL_open,AAL_high,AAL_low,AAL_close,AAL_volume,AAL_return,AAL_log_return,AAL_expected_return,...,XOM_open,XOM_high,XOM_low,XOM_close,XOM_volume,XOM_return,XOM_log_return,XOM_expected_return,XOM_abnormal_return,XOM_typical_price
0,-60,2017-09-22 16:55:00+00:00,47.290,47.290,47.280,47.280,2403.0,-0.000002,-0.000002,0.000081,...,79.7900,79.7900,79.7700,79.7888,16825.0,-0.000099,-0.000099,-0.000085,-0.000014,79.782933
1,-59,2017-09-22 16:56:00+00:00,47.280,47.280,47.250,47.251,5165.0,-0.000613,-0.000614,0.000081,...,79.7813,79.8100,79.7800,79.8100,23716.0,0.000266,0.000266,-0.000085,0.000350,79.800000
2,-58,2017-09-22 16:57:00+00:00,47.250,47.275,47.250,47.270,10170.0,0.000402,0.000402,0.000068,...,79.8000,79.8100,79.7900,79.7910,20418.0,-0.000238,-0.000238,-0.000077,-0.000161,79.797000
3,-57,2017-09-22 16:58:00+00:00,47.270,47.300,47.260,47.295,7561.0,0.000529,0.000529,0.000081,...,79.7950,79.8150,79.7947,79.8000,19244.0,0.000113,0.000113,-0.000083,0.000196,79.803233
4,-56,2017-09-22 16:59:00+00:00,47.300,47.310,47.265,47.270,20136.0,-0.000529,-0.000529,0.000083,...,79.8000,79.8000,79.7900,79.7900,6980.0,-0.000125,-0.000125,-0.000085,-0.000040,79.793333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,56,2017-09-22 18:51:00+00:00,47.170,47.190,47.165,47.190,5978.0,0.000318,0.000318,-0.000044,...,79.7900,79.7910,79.7800,79.7856,7209.0,-0.000045,-0.000045,0.000002,-0.000047,79.785533
117,57,2017-09-22 18:52:00+00:00,47.185,47.200,47.180,47.180,6283.0,-0.000212,-0.000212,-0.000049,...,79.7850,79.8000,79.7800,79.7990,9495.0,0.000168,0.000168,0.000003,0.000165,79.793000
118,58,2017-09-22 18:53:00+00:00,47.185,47.210,47.185,47.200,5426.0,0.000424,0.000424,-0.000058,...,79.8000,79.8100,79.7950,79.8000,10061.0,0.000013,0.000013,0.000008,0.000004,79.801667
119,59,2017-09-22 18:54:00+00:00,47.210,47.210,47.205,47.205,2442.0,0.000106,0.000106,-0.000046,...,79.8001,79.8001,79.7900,79.7950,13881.0,-0.000063,-0.000063,0.000008,-0.000071,79.795033


In [10]:
# Build panels for ALL tweets in the filtered DataFrame and save them to the specified directory.

# NOTE: Only doing 3 as example for submission, since this takes approx 1 minute per tweet.
# In practice we split this among several computers to cut down total runtime.
df_tweets_filtered_filtered = df_tweets_filtered.iloc[:3]

for id in df_tweets_filtered_filtered["id"].values:  # replace with df_tweets_filtered for full set
    print("Processing tweet id:", id)
    panel, info = build_minutely_event_panel(
        tweet_id=id,
        df_tweets=df_tweets_filtered_filtered,  # replace with df_tweets_filtered for full set
        event_ts_col="date_utc",
        minutely_dir="DATA MINUTELY", session_start_date=date(2015,12,1),
        save_csv=True
    )

    print(info, "\n")

Processing tweet id: 911287725847908352
Saved panel for tweet 911287725847908352 to MINUTELY EVENT STUDY PANELS BY COMPANY\911287725847908352_minutely_panel.csv
{'tweet_id': np.int64(911287725847908352), 'event_ts': '2017-09-22T17:54:59+00:00', 'session_start_date': '2017-09-15', 'session_end_date': '2017-09-29', 'n_tickers_processed': 49, 'n_tickers_skipped': 0, 'skipped': [], 'n_rows': 121, 't_range': (-60, 60), 'saved_to': 'MINUTELY EVENT STUDY PANELS BY COMPANY\\911287725847908352_minutely_panel.csv'} 

Processing tweet id: 1184147319480041473
Saved panel for tweet 1184147319480041473 to MINUTELY EVENT STUDY PANELS BY COMPANY\1184147319480041473_minutely_panel.csv
{'tweet_id': np.int64(1184147319480041473), 'event_ts': '2019-10-15T16:41:36+00:00', 'session_start_date': '2019-10-08', 'session_end_date': '2019-10-22', 'n_tickers_processed': 49, 'n_tickers_skipped': 0, 'skipped': [], 'n_rows': 121, 't_range': (-60, 60), 'saved_to': 'MINUTELY EVENT STUDY PANELS BY COMPANY\\118414731948

### Per Industry

In [11]:
def build_industry_panel_from_company_panel_returns_only(
    panel_df: pd.DataFrame,
    mapping_path: str = None,
    mapping_df: pd.DataFrame = None,
    industry_col_name: str = "Industry",
    ticker_col_name: str = "Ticker",
    how_return: str = "equal"   # "equal" or "volume"
) -> pd.DataFrame:
    """
    Aggregate company-level panel into industry-level panel (returns only).

    Behavior changes from previous version:
      - Always preserves 't' and 'timestamp' columns from the input (if present).
      - Ignores tickers that are not present in the mapping (they will NOT produce NaN columns).
      - Only produces industry columns for industries that have >=1 mapped ticker with a detected return column.
    """
    if mapping_df is None and mapping_path is None:
        raise ValueError("Provide mapping_path or mapping_df (ticker -> industry).")

    # Load mapping if needed
    if mapping_df is None:
        if mapping_path.lower().endswith((".xls", ".xlsx")):
            mapping_df = pd.read_excel(mapping_path)
        else:
            mapping_df = pd.read_csv(mapping_path)

    # normalize tickers in mapping
    mapping_df[ticker_col_name] = mapping_df[ticker_col_name].astype(str).str.strip()

    # 1) detect tickers by scanning columns for return/log_return patterns
    cols = list(panel_df.columns)
    # Common return suffixes/tokens (expand if you have other naming conventions)
    preferred_suffixes = ["_log_return", "_daily_return", "_return", "_ret", "_r"]
    detected_tickers = set()

    # detection: prefer exact suffix matches first
    for c in cols:
        for suf in preferred_suffixes:
            if c.endswith(suf):
                detected_tickers.add(c[: -len(suf)])
                break
    # fallback: find columns that contain 'return' and take the prefix before first '_' or '.' or whitespace
    if not detected_tickers:
        for c in cols:
            if "return" in c.lower():
                parts = re.split(r'[_\.\s]', c)
                if parts:
                    detected_tickers.add(parts[0])

    detected_tickers = sorted(detected_tickers)

    if not detected_tickers:
        raise ValueError("No return-like columns found in panel_df. Ensure your company panel has columns with return in the name (e.g., 'AAPL_return').")

    # 2) subset mapping to only tickers present in the panel (we ignore unmapped tickers)
    mapping_sub = mapping_df[mapping_df[ticker_col_name].isin(detected_tickers)].copy()
    mapped_tickers = set(mapping_sub[ticker_col_name].tolist())

    # If nothing from mapping matches detected tickers -> return skeleton with t,timestamp but no industry columns
    if mapping_sub.empty:
        # build output skeleton preserving t and timestamp if present
        out_df = pd.DataFrame(index=panel_df.index)
        for c in ("t", "timestamp"):
            if c in panel_df.columns:
                out_df[c] = panel_df[c].values
        # no industries to compute
        return out_df

    # 3) build industry groups from mapping_sub only
    industry_groups = mapping_sub.groupby(industry_col_name)[ticker_col_name].apply(list).to_dict()

    # 4) prepare output skeleton, preserve 't' and 'timestamp' if present
    out_df = pd.DataFrame(index=panel_df.index)
    for c in ("t", "timestamp"):
        if c in panel_df.columns:
            out_df[c] = panel_df[c].values

    # helper to find candidate columns for a ticker
    def find_return_cols_for_ticker(t):
        found_ret = []
        found_log = []
        # check exact candidates first
        for suf in preferred_suffixes:
            cand = f"{t}{suf}"
            if cand in panel_df.columns:
                if "log" in suf:
                    found_log.append(cand)
                else:
                    found_ret.append(cand)
        # also accept any column that starts with ticker and contains 'return' if not found yet
        if not found_ret and not found_log:
            matches = [c for c in panel_df.columns if c.startswith(f"{t}") and "return" in c.lower()]
            for c in matches:
                if "log" in c.lower():
                    found_log.append(c)
                else:
                    found_ret.append(c)
        return found_ret, found_log

    # 5) compute industry returns only for industries that actually have at least 1 mapped ticker
    for industry, tickers in industry_groups.items():
        # keep only tickers that are both mapped and present in panel detected_tickers
        tickers_present = [t for t in tickers if t in mapped_tickers]
        if not tickers_present:
            # skip industries with no present mapped tickers
            continue

        # build lists of columns for returns and volumes
        ret_cols = []
        log_cols = []
        vol_cols = []
        for t in tickers_present:
            found_ret, found_log = find_return_cols_for_ticker(t)
            ret_cols += found_ret
            log_cols += found_log
            # look for volume variants
            for v_suf in ("_daily_volume", "_volume", "_vol"):
                vcol = f"{t}{v_suf}"
                if vcol in panel_df.columns:
                    vol_cols.append(vcol)
        # deduplicate column lists while preserving order
        def dedup_keep_order(seq):
            seen = set(); out=[]
            for x in seq:
                if x not in seen:
                    seen.add(x); out.append(x)
            return out
        ret_cols = dedup_keep_order(ret_cols)
        log_cols = dedup_keep_order(log_cols)
        vol_cols = dedup_keep_order(vol_cols)

        # If no return columns found for this industry's mapped tickers -> skip
        if not ret_cols and not log_cols:
            continue

        rets_df = panel_df[ret_cols].astype(float) if ret_cols else pd.DataFrame(index=panel_df.index)
        logs_df = panel_df[log_cols].astype(float) if log_cols else pd.DataFrame(index=panel_df.index)
        vols_df = panel_df[vol_cols].astype(float) if vol_cols else pd.DataFrame(index=panel_df.index)

        # compute industry return (prefer raw returns; if only log returns exist you can compute on logs too)
        if how_return == "equal":
            if not rets_df.empty:
                industry_return = rets_df.mean(axis=1, skipna=True)
            elif not logs_df.empty:
                industry_return = logs_df.mean(axis=1, skipna=True)
            else:
                industry_return = pd.Series([np.nan]*len(panel_df), index=panel_df.index)
        elif how_return == "volume":
            # volume-weighted: require both volumes and returns
            if vols_df.empty or rets_df.empty:
                # if volumes or returns missing -> produce NaN series (skip)
                industry_return = pd.Series([np.nan]*len(panel_df), index=panel_df.index)
            else:
                vol_sum = vols_df.fillna(0).sum(axis=1)
                ret_num = (rets_df.fillna(0) * vols_df.fillna(0)).sum(axis=1)
                industry_return = ret_num.div(vol_sum).where(vol_sum != 0, np.nan)
        else:
            raise ValueError("how_return must be 'equal' or 'volume'")

        # add to output only if we have non-empty industry_return (i.e., was computable)
        if industry_return.notna().any():
            safe_ind = re.sub(r'[^0-9A-Za-z_]', '_', str(industry))
            out_df[f"{safe_ind}_return"] = industry_return.values
        # else: skip adding column (no valid data)

    return out_df

In [12]:
INPUT_DIR_COMPANY = "MINUTELY EVENT STUDY PANELS BY COMPANY"
OUTPUT_DIR_INDUSTRY = "MINUTELY EVENT STUDY PANELS BY INDUSTRY"
MAPPING_PATH = "tickers.xlsx"
HOW_RETURN = "equal"  # or "volume"

os.makedirs(OUTPUT_DIR_INDUSTRY, exist_ok=True)
panel_files = sorted(glob.glob(os.path.join(INPUT_DIR_COMPANY, "*.csv")))

print(f"About to process {len(panel_files)} CSV files (company level -> industry aggregate)")

for file_path in panel_files:
    fname = os.path.basename(file_path)
    print(f"Processing {fname} ...")
    try:
        panel_df = pd.read_csv(file_path)
    except Exception as e:
        print(f"  Skipping (read error): {e}")
        continue

    try:
        df_ind = build_industry_panel_from_company_panel_returns_only(
            panel_df=panel_df,
            mapping_path=MAPPING_PATH,
            industry_col_name="Industry",
            ticker_col_name="Ticker",
            how_return=HOW_RETURN
        )
    except Exception as e:
        print(f"  Skipping (error during aggregation): {e}")
        continue

    # if df_ind contains only t/timestamp (no industry return columns) you may want to skip saving
    # decide policy: here we save but you can skip by checking column count
    if list(df_ind.columns) == ["t"] or list(df_ind.columns) == ["timestamp"] or (set(df_ind.columns) <= {"t","timestamp"}):
        print("  No mapped tickers with return columns found for this file -> skipping saving.")
        continue

    out_name = fname.replace("panel", "industry_panel")
    out_path = os.path.join(OUTPUT_DIR_INDUSTRY, out_name)
    df_ind.to_csv(out_path, index=False)
    print(f"  Saved {out_path}")
print("Done.")

About to process 228 CSV files (company level -> industry aggregate)
Processing 1000391997969092608_minutely_panel.csv ...
  Saved MINUTELY EVENT STUDY PANELS BY INDUSTRY\1000391997969092608_minutely_industry_panel.csv
Processing 1010503423773507584_minutely_panel.csv ...
  Saved MINUTELY EVENT STUDY PANELS BY INDUSTRY\1010503423773507584_minutely_industry_panel.csv
Processing 1015586529484443648_minutely_panel.csv ...
  Saved MINUTELY EVENT STUDY PANELS BY INDUSTRY\1015586529484443648_minutely_industry_panel.csv
Processing 1019932691339399168_minutely_panel.csv ...
  Saved MINUTELY EVENT STUDY PANELS BY INDUSTRY\1019932691339399168_minutely_industry_panel.csv
Processing 1021384752136409088_minutely_panel.csv ...
  Saved MINUTELY EVENT STUDY PANELS BY INDUSTRY\1021384752136409088_minutely_industry_panel.csv
Processing 1021917767467982854_minutely_panel.csv ...
  Saved MINUTELY EVENT STUDY PANELS BY INDUSTRY\1021917767467982854_minutely_industry_panel.csv
Processing 1023546197129224192_

### Filtering only mentioned industries return 

In [13]:
# Get files
industry_panel_files = sorted(glob.glob(os.path.join(OUTPUT_DIR_INDUSTRY, "*.csv")))
df_panel_paths = pd.DataFrame({"panel_path": industry_panel_files})

# robust extractor: first try tweet_id_123, then last integer group
def extract_id_from_filename(path):
    base = os.path.splitext(os.path.basename(path))[0]
    m = re.search(r'tweet_id[_\-]?(\d+)', base, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m2 = re.findall(r'(\d+)', base)
    if m2:
        return int(m2[-1])
    return None

df_panel_paths["id_extracted"] = df_panel_paths["panel_path"].apply(extract_id_from_filename)

# report problematic filenames
bad = df_panel_paths[df_panel_paths["id_extracted"].isna()]
if not bad.empty:
    print("Warning: couldn't parse id for these files:")
    for p in bad["panel_path"].tolist():
        print("  ", p)

# store as pandas nullable integer type (keeps NaNs)
df_panel_paths["id"] = df_panel_paths["id_extracted"].astype("Int64")

# Now ensure df_tweets_filtered.id is same dtype
# (df_tweets_filtered comes from your environment)
df_tweet_id_industries = df_tweets_filtered[["id", "industry"]].copy()
# convert to Int64 to match
df_tweet_id_industries["id"] = df_tweet_id_industries["id"].astype("Int64")

# Merge
df_tweet_id_industries = df_tweet_id_industries.merge(
    df_panel_paths[["panel_path", "id"]],
    on="id",
    how="left",
)

display(df_tweet_id_industries.head())
print("Missing panel_path:", df_tweet_id_industries["panel_path"].isna().sum())

,id,industry,panel_path
0,911287725847908352,Airlines & Travel,MINUTELY EVENT STUDY PANELS BY INDUSTRY\911287...
1,1184147319480041473,Airlines & Travel,MINUTELY EVENT STUDY PANELS BY INDUSTRY\118414...
2,1184631273454817280,Airlines & Travel,MINUTELY EVENT STUDY PANELS BY INDUSTRY\118463...
3,1184987864125321216,Airlines & Travel,MINUTELY EVENT STUDY PANELS BY INDUSTRY\118498...
4,700795170023825408,Technology & Internet,MINUTELY EVENT STUDY PANELS BY INDUSTRY\700795...


Missing panel_path: 0


In [14]:
# Specify output directory and ensure it exists
OUTPUT_DIR_INDUSTRY_SINGLE = "MINUTELY EVENT STUDY PANELS BY SINGLE INDUSTRY"
os.makedirs(OUTPUT_DIR_INDUSTRY_SINGLE, exist_ok=True)

# Optional manual overrides for weird industry names -> column names
# (keep keys as the raw industry string present in df_tweet_id_industries)
SPECIAL_COLNAME_MAP = {
    "Chemicals": "Chemicals_(Materials_sector)_minutely_return",
    "Semiconductors": "Semiconductors_(Information_Technology_sector)_minutely_return",
}

# Suffix candidates we'll try when looking for the industry return column
RETURN_SUFFIX_CANDIDATES = [
    "_minutely_return",
    "_minute_return",
    "_return",
    "_log_return",
    "_minutely_log_return",
]

# Helper: produce safe industry prefix used in column names
def safe_ind_prefix(industry_str: str) -> str:
    return str(industry_str).replace(" ", "_").replace("/", "_")

# Helper: find a best-matching return column name in df.columns for a given industry
def find_industry_return_column(df_columns, industry):
    cols_set = set(df_columns)

    # 1) check explicit special override
    if industry in SPECIAL_COLNAME_MAP:
        cand = SPECIAL_COLNAME_MAP[industry]
        if cand in cols_set:
            return cand

    # 2) try exact safe prefixes + known suffixes
    safe = safe_ind_prefix(industry)
    for suf in RETURN_SUFFIX_CANDIDATES:
        cand = f"{safe}{suf}"
        if cand in cols_set:
            return cand

    # 3) sometimes the industry name might include parentheses or underscores already
    # try the raw industry (with suffixes)
    for suf in RETURN_SUFFIX_CANDIDATES:
        cand = f"{industry}{suf}"
        if cand in cols_set:
            return cand

    # 4) fallback: find any column that contains all (normalized) words of industry and 'return'
    # e.g., "Airlines & Travel" -> match a column containing "airlines" and "travel" and "return"
    tokens = re.findall(r"[A-Za-z0-9]+", industry.lower())
    if tokens:
        for col in df_columns:
            col_l = col.lower()
            if "return" not in col_l:
                continue
            if all(tok in col_l for tok in tokens):
                return col

    # 5) last resort: any column that endswith 'return' and contains at least one token
    for col in df_columns:
        col_l = col.lower()
        if col_l.endswith("return") and any(tok in col_l for tok in tokens):
            return col

    # not found
    return None

# Iterate and create one-file-per-tweet/industry
saved = 0
skipped = 0
for idx, row in df_tweet_id_industries.iterrows():
    tweet_id = row["id"]
    industry = row["industry"]

    # read panel path
    panel_path = row.get("panel_path")
    if not panel_path or not os.path.exists(panel_path):
        print(f"[{idx}] Skipping tweet {tweet_id}: panel_path missing or not found -> {panel_path}")
        skipped += 1
        continue

    try:
        df = pd.read_csv(panel_path)
    except Exception as e:
        print(f"[{idx}] Skipping tweet {tweet_id} (read error): {e}")
        skipped += 1
        continue

    # sanity: require at least one of `t` or `timestamp`
    keep_cols = []
    if "t" in df.columns:
        keep_cols.append("t")
    if "timestamp" in df.columns:
        keep_cols.append("timestamp")
    if len(keep_cols) == 0:
        print(f"[{idx}] Skipping tweet {tweet_id}: panel missing both 't' and 'timestamp' columns.")
        skipped += 1
        continue

    # find the appropriate industry return column
    ret_col = find_industry_return_column(df.columns, industry)
    if ret_col is None:
        print(f"[{idx}] Skipping tweet {tweet_id}: could not find a return column for industry '{industry}' in {panel_path}")
        skipped += 1
        continue

    # Select only t/timestamp and the found return column
    out_df = df[keep_cols + [ret_col]].copy()

    # Fill NaNs in return column with 0.0 as you did before
    out_df[ret_col] = out_df[ret_col].fillna(0.0)

    # Create a tidy column name for output: <SAFEIND>_minutely_return
    safe = safe_ind_prefix(industry)
    out_col_name = f"{safe}_minutely_return"
    # rename the found column to the tidy column name
    out_df = out_df.rename(columns={ret_col: out_col_name})

    # Build output filename: replace the first occurrence of "industry" in the original filename
    base_name = os.path.basename(panel_path)
    if "industry" in base_name:
        new_name = base_name.replace("industry", industry[:4].strip(), 1)
    else:
        # fallback: prefix with tweet id
        new_name = f"{tweet_id}_" + base_name

    out_path = Path(OUTPUT_DIR_INDUSTRY_SINGLE) / new_name

    try:
        out_df.to_csv(out_path, index=False)
        saved += 1
    except Exception as e:
        print(f"[{idx}] Failed to write {out_path}: {e}")
        skipped += 1
        continue

print(f"Finished. Saved {saved} files, skipped {skipped}.")

Finished. Saved 238 files, skipped 0.
